# SynnoDB: From DuckDB to Bespoke in One Import

SynnoDB is a drop-in replacement for DuckDB that transparently accelerates your SQL queries
with auto-generated bespoke C++ engines - while falling back to DuckDB for everything else.
No schema changes. No query rewrites. One import.

This notebook walks through the full journey for **TPC-H Q1-Q5**:

1. **Baseline** - run Q1-Q5 on vanilla DuckDB at SF20, 10 parameter instantiations each.
2. **Generate** - point SynnoDB at a `queries.json` file and let it write the engine.
3. **Drop in** - replace one import, re-run the identical queries, compare the numbers.

> **Prerequisites** - see [`docs/TUTORIAL_base_implementation.md`](../docs/TUTORIAL_base_implementation.md)
> for installation, TPC-H data generation, and model endpoint setup.

## Setup

Adjust the paths below if your data lives elsewhere.

In [ ]:
import os, json, time, statistics
from pathlib import Path

DATA_ROOT   = Path(os.environ.get("SYNNO_DATA_DIR",  "/mnt/labstore/learneddb/synno_data"))
PARQUET_DIR = Path(os.environ.get("SYNNO_TPCH_PARQUET",
                   DATA_ROOT / "workloads/tpch/tpch_parquet"))
ENGINES_DIR = Path(os.environ.get("SYNNO_ENGINES_DIR", DATA_ROOT / "engines"))
MODEL       = os.environ.get("SYNNO_MODEL", "anthropic/claude-sonnet-4-6") # e.g. "anthropic/claude-sonnet-4-6", "gpt-5.4", "openai/unsloth/MiniMax-M3"
SF          = 20
TABLES      = ["customer", "lineitem", "nation", "orders", "part",
               "partsupp", "region", "supplier"]

ENGINES_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("SYNNO_ENGINES_DIR", str(ENGINES_DIR))
print("Parquet root :", PARQUET_DIR)
print("Engines dir  :", ENGINES_DIR)
print("Model        :", MODEL)

Parquet root : /mnt/labstore/learneddb/synno_data/workloads/tpch/tpch_parquet
Engines dir  : /mnt/labstore/learneddb/synno_data/engines
Model        : anthropic/claude-sonnet-4-6


---
## Step 1 - DuckDB Baseline

We run **Q1-Q5 on vanilla DuckDB** at SF20: 10 instantiations per query
(different placeholder values, drawn from the actual data), total wall-clock time recorded.
These exact SQL strings will be reused in Step 3 so the comparison is apples-to-apples.

### Register the BYO workload

The workload is described by a single self-describing JSON file. Each entry carries its SQL
template **and** the values for its `[PLACEHOLDER]` slots - one key per placeholder mapping
to the list of values it should take across the sweep:

```json
"7": {
  "sql": "... n1.n_name = '[NATION1]' ... n2.n_name = '[NATION2]' ...",
  "params": { "NATION1": ["GERMANY", "CHINA"], "NATION2": ["ROMANIA", "UNITED STATES"] }
}
```

`register_workload_from_json` reads it and derives the schema from the parquet via DuckDB.
Per-placeholder lists are index-zipped into instantiations (so correlated placeholders stay
aligned); a length-1 list broadcasts across the sweep. This is the shape a BI dashboard would
fill - a slider per placeholder predefining its values.

In [2]:
import random
from synnodb.workloads.byo_workload import register_workload_from_json

QUERIES_JSON = Path("queries.json")   # lives next to this notebook

spec = register_workload_from_json(
    name="tpch_byo",
    queries_json=QUERIES_JSON,
    parquet_dir=PARQUET_DIR,
    scale_factors=(1, 2, SF),
    schema_example_table="lineitem",
)

print("Workload :", spec.name)
print("Tables   :", spec.tables)
print("Queries  :", spec.all_query_ids)

Workload : tpch_byo
Tables   : ('customer', 'lineitem', 'nation', 'orders', 'part', 'partsupp', 'region', 'supplier')
Queries  : ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22')


Here is what the queries look like - SQL templates with `[PLACEHOLDER]` slots, plus the
values supplied for them:

In [3]:
queries = json.loads(QUERIES_JSON.read_text())
for qid, entry in queries.items():
    print(f"=== Q{qid} ===")
    print(entry["sql"][:240], "...")
    print("params:", entry.get("params", {}))
    print()

=== Q1 ===
select 
    l_returnflag,  
    l_linestatus,  
    sum(l_quantity) as sum_qty, 
    sum(l_extendedprice) as sum_base_price, 
    sum(l_extendedprice*(1-l_discount)) as sum_disc_price, 
    sum(l_extendedprice*(1-l_discount)*(1+l_tax)) as s ...
params: {'DELTA': ['97', '114', '72', '118', '94', '119']}

=== Q2 ===
select
    s_acctbal,
    s_name,
    n_name,
    p_partkey,
    p_mfgr,
    s_address,
    s_phone,
    s_comment
from
    part,
    supplier,
    partsupp,
    nation,
    region
where
    p_partkey = ps_partkey
    and s_suppkey = ps_sup ...
params: {'SIZE': ['10', '11', '12', '40', '38', '35'], 'TYPE': ['COPPER', 'COPPER', 'STEEL', 'STEEL', 'TIN', 'COPPER'], 'REGION': ['ASIA', 'MIDDLE EAST', 'ASIA', 'MIDDLE EAST', 'MIDDLE EAST', 'MIDDLE EAST']}

=== Q3 ===
select l_orderkey,  
    sum(l_extendedprice*(1-l_discount)) as revenue, 
    o_orderdate,  
    o_shippriority 
FROM
    customer,  
    orders,  
    lineitem 
WHERE
    c_mktsegment = '[SEGMENT]' 
    and 

### Draw parameter instantiations

`query_gen_factory` fills the templates from the supplied value lists. We draw with a fixed
seed so the instantiations are **identical** for the DuckDB and SynnoDB runs.

In [4]:
N_REPS = 10
rng    = random.Random(42)
gen    = spec.query_gen_factory(None)

# gen(query_name, rng) -> (query_name, sql_with_values, params_dict)
instantiations = {}
for qid in spec.all_query_ids:
    instantiations[qid] = [gen(f"Q{qid}", rng) for _ in range(N_REPS)]

print(f"Drew {N_REPS} instantiations per query.")
for qid, insts in instantiations.items():
    sample_params = [i[2] for i in insts[:2]]
    print(f"  Q{qid}: {sample_params} ...")

Drew 10 instantiations per query.
  Q1: [{'DELTA': '119'}, {'DELTA': '97'}] ...
  Q2: [{'SIZE': '35', 'TYPE': 'COPPER', 'REGION': 'MIDDLE EAST'}, {'SIZE': '35', 'TYPE': 'COPPER', 'REGION': 'MIDDLE EAST'}] ...
  Q3: [{'SEGMENT': 'BUILDING', 'DATE': '1995-03-23'}, {'SEGMENT': 'AUTOMOBILE', 'DATE': '1995-03-21'}] ...
  Q4: [{'DATE': '1996-11-01'}, {'DATE': '1997-08-01'}] ...
  Q5: [{'REGION': 'AFRICA', 'DATE': '1995-01-01'}, {'REGION': 'MIDDLE EAST', 'DATE': '1993-01-01'}] ...
  Q6: [{'DATE': '1994-01-01', 'DISCOUNT': '0.09', 'QUANTITY': '25'}, {'DATE': '1994-01-01', 'DISCOUNT': '0.09', 'QUANTITY': '25'}] ...
  Q7: [{'NATION1': 'UNITED KINGDOM', 'NATION2': 'VIETNAM'}, {'NATION1': 'EGYPT', 'NATION2': 'JORDAN'}] ...
  Q8: [{'NATION': 'JAPAN', 'REGION': 'AMERICA', 'TYPE': 'STANDARD PLATED NICKEL'}, {'NATION': 'ETHIOPIA', 'REGION': 'AMERICA', 'TYPE': 'PROMO PLATED NICKEL'}] ...
  Q9: [{'COLOR': 'frosted'}, {'COLOR': 'orange'}] ...
  Q10: [{'DATE': '1994-05-01'}, {'DATE': '1994-12-01'}] ...
  

### Run on DuckDB

In [5]:
import duckdb
from tqdm import tqdm

sf_dir = PARQUET_DIR / f"sf{SF}"

duck = duckdb.connect(":memory:")
for t in TABLES:
    duck.execute(
        f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{sf_dir}/{t}.parquet')"
    )

baseline_times = {}
for qid, insts in tqdm(instantiations.items(), desc="Running baseline queries"):
    times = []
    for _, sql, _ in insts:
        t0 = time.perf_counter()
        duck.execute(sql).fetchall()
        times.append((time.perf_counter() - t0) * 1_000)
    baseline_times[qid] = times

duck.close()

total_duck = sum(sum(v) for v in baseline_times.values())
print(f"{'Query':<8} {'Avg (ms)':>12} {'Median (ms)':>14}")
print("-" * 38)
for qid in spec.all_query_ids:
    t = baseline_times[qid]
    print(f"Q{qid:<7} {sum(t)/len(t):>12.1f} {statistics.median(t):>14.1f}")
print("-" * 38)
print(f"{'TOTAL':<8} {total_duck:>12.1f}")

Running baseline queries: 100%|██████████| 22/22 [00:33<00:00,  1.52s/it]

Query        Avg (ms)    Median (ms)
--------------------------------------
Q1              123.1          120.2
Q2              105.2          108.0
Q3              138.8          135.8
Q4              123.6          121.9
Q5              173.4          169.7
Q6               66.1           65.2
Q7              116.1          115.9
Q8              151.0          148.5
Q9              331.9          318.3
Q10             188.3          189.4
Q11              48.3           47.6
Q12             112.7          111.8
Q13             140.1          136.6
Q14              94.3           93.9
Q15              98.0           98.4
Q16             146.5          147.6
Q17             131.2          130.5
Q18             266.8          269.1
Q19             112.7          109.2
Q20             135.4          134.6
Q21             392.0          392.3
Q22             141.5          148.8
--------------------------------------
TOTAL         33369.8


---
## Step 2 - Generate the SynnoDB Engine

You hand SynnoDB the same `queries.json` and a scale factor. It:

1. **Creates a storage plan** - decides how each query's columns are laid out in memory.
2. **Implements the engine** - writes single-threaded C++, compiles it, validates every output
   against DuckDB, then **auto-publishes** the binary into `ENGINES_DIR`.

This is a one-time cost. Once published the engine is discovered automatically across sessions.

### Storage plan

In [9]:
from synnodb import SynnoDB

db   = SynnoDB(workload="tpch_byo", model=MODEL, db_storage="in_memory", queries="1-5", data_dir=DATA_ROOT)
plan = db.createStoragePlan()

print("Run :", plan.run_id)
print()
print(plan.text[:600], "...")

Output()

weave: weave version 0.53.0 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
2026-06-30 07:07:16 INFO:weave.trace.init_message:weave version 0.53.0 is available!  To upgrade, please run:
 $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: jwehrstein.
weave: View Weave data at https://wandb.ai/learneddb/bespoke-olap-internal/weave
2026-06-30 07:07:16 INFO:weave.trace.init_message:Logged in as Weights & Biases user: jwehrstein.
View Weave data at https://wandb.ai/learneddb/bespoke-olap-internal/weave
2026-06-30 07:07:17 DEBUG:git.util:sys.platform='linux', git_executable='git'
2026-06-30 07:07:17 DEBUG:git.cmd:Popen(['git', 'cat-file', '--batch-check'], cwd=/home/jwehrstein/SynnoDB, stdin=<valid stream>, shell=False, universal_newlines=False)


wandb: Initializing weave.


Output()

weave: weave version 0.53.0 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
2026-06-30 07:07:18 INFO:weave.trace.init_message:weave version 0.53.0 is available!  To upgrade, please run:
 $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: jwehrstein.
weave: View Weave data at https://wandb.ai/learneddb/bespoke-olap-internal/weave
2026-06-30 07:07:18 INFO:weave.trace.init_message:Logged in as Weights & Biases user: jwehrstein.
View Weave data at https://wandb.ai/learneddb/bespoke-olap-internal/weave
2026-06-30 07:07:18 DEBUG:asyncio:Using selector: EpollSelector
2026-06-30 07:07:18 INFO:synnodb.main:Using database source: DBStorage.IN_MEMORY. Disk DB dir: None
2026-06-30 07:07:18 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 0 read-only artifact files to `/home/jwehrstein/SynnoDB/tutorials/output` for benchmark tpch_byo
2026-06-30 07:07:18 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 1 artifact files to

apply_patch/added_loc_count,▁
apply_patch/create_count,▁
apply_patch/deleted_loc_count,▁
cached_tokens,▁▅███
code/loc,▁▁▁▁▁▁▁▁▁
context_window_usage,▁▂▂██
cost_usd,▁▁▁█▃
final/num_prompts,▁
final/total_cost_usd,▁
final/total_real_cost_usd,▁
+23,...


Run : ttl37hbj

  IN-MEMORY STORAGE LAYOUT PLAN
  Workload: TPC-H Queries 1–5
  Target:   Single-node, analytical, Arrow-ingested typed columns

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WORKLOAD ANALYSIS SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Query 1  – Pure lineitem scan + group-by on (l_returnflag, l_linestatus).
           Heavy aggregation: sum/avg of l_quantit ...


### Base implementation

We feed the plan **content** straight in via `storage_plan=plan.text`, so this step needs
no W&B. If you instead chain off a logged storage-plan run, pass its run id with
`db.createBaseImpl(storage_plan_wandb_id=plan.run_id)`. Provide exactly one of the two.

In [ ]:
impl = db.createBaseImpl(storage_plan=plan.text)  # pass the plan content directly (W&B-free)

print("Workspace :", impl.workspace)
print("Files     :", sorted(impl.files))
print()
print(f"Engine published to: {ENGINES_DIR}")

---
## Step 3 - Drop In SynnoDB

The only change is **one import line** and two extra keyword arguments to `connect()`:

```diff
- import duckdb
+ import synnodb as duckdb
+ from synnodb.router import RouterMode, RouterPolicy

  con = duckdb.connect(
      ":memory:",
+     engines=str(ENGINES_DIR),
+     policy=RouterPolicy(mode=RouterMode.SAMPLED, cross_check_rate=1.0),
  )
```

Every other line - the view setup, the `execute()` calls, `fetchall()` - is identical.

### Open the drop-in connection

In [ ]:
import synnodb as duckdb                          # <- only change
from synnodb.router import RouterMode, RouterPolicy

con = duckdb.connect(
    ":memory:",
    engines=str(ENGINES_DIR),
    policy=RouterPolicy(mode=RouterMode.SAMPLED, cross_check_rate=1.0),
)

sf_dir = PARQUET_DIR / f"sf{SF}"
for t in TABLES:
    con.duckdb.execute(
        f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{sf_dir}/{t}.parquet')"
    )

con.refresh_engines()
n = con.router_stats()["registry"]["templates"]
print(f"Discovered {n} engine template(s) under {ENGINES_DIR}")

### Run the same queries with the same parameter values

We iterate over `instantiations` - the exact SQL strings from Step 1.

In [ ]:
synno_times = {}
for qid, insts in instantiations.items():
    times = []
    for _, sql, _ in insts:
        t0 = time.perf_counter()
        con.execute(sql).fetchall()
        times.append((time.perf_counter() - t0) * 1_000)
    synno_times[qid] = times

### Speedup

In [ ]:
total_synno = sum(sum(v) for v in synno_times.values())

print(f"{'Query':<8} {'DuckDB (ms)':>12} {'SynnoDB (ms)':>14} {'Speedup':>9}")
print("-" * 48)
for qid in spec.all_query_ids:
    d = sum(baseline_times[qid])
    s = sum(synno_times[qid])
    speedup = d / s if s > 0 else float("inf")
    print(f"Q{qid:<7} {d:>12.1f} {s:>14.1f} {speedup:>8.2f}x")
print("-" * 48)
overall = total_duck / total_synno if total_synno > 0 else float("inf")
print(f"{'TOTAL':<8} {total_duck:>12.1f} {total_synno:>14.1f} {overall:>8.2f}x")

### Correctness guarantee

Every routed result was compared against DuckDB (`cross_check_rate=1.0`).
The mismatch count must be 0.

In [ ]:
stats = con.router_stats()["session"]
print(f"Routed:         {stats['routed']}")
print(f"Cross-checked:  {stats['cross_checked']}")
print(f"Mismatches:     {stats['cross_check_mismatch']}")
assert stats["cross_check_mismatch"] == 0, "result divergence detected!"
print("\nAll results match DuckDB exactly.")

### Q1 result (with timing footer)

In [ ]:
_, q1_sql, _ = instantiations["1"][0]
con.execute(q1_sql)
con.show()
con.close()

---
## Where to go next

The base implementation is single-threaded. The same `SynnoDB` object carries each engine further:

```python
opt   = db.runOptimLoop(base_impl=impl)           # single-threaded SIMD / cache optimization
multi = db.addMultiThreading(optimized=opt)        # parallel execution across cores
rep   = db.checkSfCorrectness(source=multi, target_sf=50)  # correctness at larger SF
```

CLI equivalents and step-by-step commentary are in
[`docs/TUTORIAL_base_implementation.md`](../docs/TUTORIAL_base_implementation.md).